# Grandpa's own voice (Piper fine-tune)

Trains a **local, offline** Piper voice from the Achchuu recordings, so Grandpa speaks in Captain's voice without ElevenLabs.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

1. Zip `D:\GrandpaVoice\achchuu` (the folder with `wavs` and `metadata.csv`) as `achchuu.zip`, and upload it to **Google Drive → My Drive → GrandpaVoice/**.
2. Run all cells (Runtime → Run all). Every epoch saves to Drive, so if Colab disconnects, just **Run all** again and it continues.
3. When the last cell prints `DONE`, download `grandpa-achchuu.onnx` and `grandpa-achchuu.onnx.json` from Drive → GrandpaVoice/out.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/GrandpaVoice'
os.makedirs(f'{DRIVE}/ckpt', exist_ok=True); os.makedirs(f'{DRIVE}/out', exist_ok=True)
assert os.path.exists(f'{DRIVE}/achchuu.zip'), 'Upload achchuu.zip to My Drive/GrandpaVoice first'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%bash
set -e
apt-get -qq update && apt-get -qq install -y espeak-ng > /dev/null
cd /content
[ -d piper1-gpl ] || git clone -q https://github.com/OHF-Voice/piper1-gpl.git
cd piper1-gpl
pip -q install -e '.[train]'
./build_monotonic_align.sh > /dev/null
echo installed

In [ ]:
%%bash
set -e
cd /content
[ -d achchuu ] || unzip -q /content/drive/MyDrive/GrandpaVoice/achchuu.zip -d /content
# the zip may contain the folder itself or its contents
[ -f achchuu/metadata.csv ] || { mkdir -p achchuu && mv wavs metadata.csv achchuu/; }
# piper1-gpl wants the file name with its extension in the first column
python3 - <<'PY'
rows = [l.rstrip('\n').split('|', 1) for l in open('/content/achchuu/metadata.csv', encoding='utf-8') if '|' in l]
import os
keep = [(i, t) for i, t in rows if os.path.exists(f'/content/achchuu/wavs/{i}.wav')]
with open('/content/achchuu/train.csv', 'w', encoding='utf-8') as f:
    for i, t in keep:
        f.write(f'{i}.wav|{t}\n')
print(len(keep), 'clips ready')
PY

In [ ]:
# Base voice to fine-tune from (Lessac, medium). Downloaded once, kept on Drive.
import os, urllib.request
BASE = f'{DRIVE}/ckpt/lessac-medium.ckpt'
if not os.path.exists(BASE):
    url = 'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt'
    urllib.request.urlretrieve(url, BASE)
print(os.path.getsize(BASE) // 2**20, 'MB')

In [ ]:
# Resume from the newest checkpoint on Drive, else start from Lessac.
import glob
mine = sorted(glob.glob(f'{DRIVE}/ckpt/achchuu-*.ckpt'), key=os.path.getmtime)
START = mine[-1] if mine else BASE
EPOCHS = 400   # extra epochs on top of Lessac: about 5-7 hours on a T4; 250 is already usable
# The Lessac checkpoint is already at epoch 2164 and fine-tuning continues its count,
# so the stop point is 2164 + EPOCHS (resuming your own checkpoint keeps the same stop point).
MAX_EPOCHS = 2164 + EPOCHS
print('starting from', START)

In [ ]:
%cd /content/piper1-gpl
!python3 -m piper.train fit \
  --data.voice_name "grandpa-achchuu" \
  --data.csv_path /content/achchuu/train.csv \
  --data.audio_dir /content/achchuu/wavs \
  --model.sample_rate 22050 \
  --data.espeak_voice "en-us" \
  --data.cache_dir /content/cache \
  --data.config_path {DRIVE}/out/config.json \
  --data.batch_size 16 \
  --trainer.max_epochs {MAX_EPOCHS} \
  --trainer.callbacks+=ModelCheckpoint \
  --trainer.callbacks.dirpath {DRIVE}/ckpt \
  --trainer.callbacks.filename "achchuu-{{epoch:04d}}" \
  --trainer.callbacks.every_n_epochs 1 \
  --trainer.callbacks.save_top_k 2 \
  --trainer.callbacks.monitor epoch \
  --trainer.callbacks.mode max \
  --ckpt_path {START}

In [ ]:
# Export the newest checkpoint for Grandpa
import glob, shutil
last = sorted(glob.glob(f'{DRIVE}/ckpt/achchuu-*.ckpt'), key=os.path.getmtime)[-1]
print('exporting', last)
!cd /content/piper1-gpl && python3 -m piper.train.export_onnx --checkpoint "{last}" --output-file {DRIVE}/out/grandpa-achchuu.onnx
shutil.copy(f'{DRIVE}/out/config.json', f'{DRIVE}/out/grandpa-achchuu.onnx.json')
!echo 'Hello Captain, this is Grandpa, speaking in your own voice, right here on your computer.' | python3 -m piper -m {DRIVE}/out/grandpa-achchuu.onnx -f {DRIVE}/out/test.wav
from IPython.display import Audio, display
display(Audio(f'{DRIVE}/out/test.wav'))
print('DONE: download grandpa-achchuu.onnx and grandpa-achchuu.onnx.json from Drive/GrandpaVoice/out')

## On your PC (PowerShell)

```powershell
Copy-Item $HOME\Downloads\grandpa-achchuu.onnx*  $HOME\.Grandpa\voices\
[Environment]::SetEnvironmentVariable('GRANDPA_PIPER_VOICE','grandpa-achchuu','User')
cd D:\Grandpa
uv run --extra avatar --extra piper --extra tamil-voice python -m grandpa avatar
```
The startup line shows `Piper (grandpa-achchuu)`. With the ElevenLabs key still set, ElevenLabs is used first and your own local voice is the fallback; to use only the local one, set `GRANDPA_AVATAR_VOICE=piper`.